In [ ]:
%load_ext autoreload
%autoreload 2
import os
import re 
import sys
import numpy as np
import pandas as pd
import xarray as xr
from os.path import join as pjoin
from tqdm.notebook import tqdm
from sklearn.metrics import mutual_info_score
import plotly.graph_objects as go
from scipy.stats import pearsonr, spearmanr, zscore
from natsort import natsorted
import statsmodels.api as sm
from statsmodels.formula.api import ols
from numpy.random import RandomState, SeedSequence, MT19937

sys.path.append('../')
import circletrack_behavior as ctb
import circletrack_neural as ctn
import place_cells as pc
import plotting_functions as pf

In [ ]:
## Settings
project_folder = ['MultiCon_Imaging']
experiment_folders = ['MultiCon_Imaging5', 'MultiCon_Imaging6', 'MultiCon_Imaging7']
dpath = f'../../{project_folder[0]}'
fig_path = f'../../../Manuscripts/MultiCon/intermediate_plots/overlap_days_15_16'
int_data = f'../../../Manuscripts/MultiCon/intermediate_plots/intermediate_data'
chance_color = '#7d7d7d'
avg_color = '#287347'
subject_color = '#7d7d7d'
ce_colors = ['#7A22BC', '#378616']
ce_colors_dict = {'Two-context': '#378616', 'Multi-context': '#7A22BC'}
symbol_dict = {'Two-context': 'x', 'Multi-context': 'circle'}
symbols_list = ['x', 'circle']
context_colors = {'A': '#00802d', 'B': '#006c79', 'C': '#004da4', 'D': '#430073'}
mouse_colors = ['midnightblue', 'darkred', 'darkorchid', 'darkturquoise']
excluded_mice = ['mc46', 'mc56'] ## missing data for the first day in one context, so didn't include in this analysis
session_list = [f'A{x}' for x in np.arange(1, 6)] + [f'B{x}' for x in np.arange(1, 6)] + [f'C{x}' for x in np.arange(1, 6)] + [f'D{x}' for x in np.arange(1, 6)]
control_list = [f'A{x}' for x in np.arange(1, 16)] + [f'B{x}' for x in np.arange(1, 6)]
day_list = [f'Day {x}' for x in np.arange(1, 21)]
bin_size = 0.06 ## size of linear position bins equivalent to 2cm-wide bins
reward_bin_size = 0.10
time_bin_size = 1 ## in seconds
velocity_thresh = 10
centroid_distance = 4
data_of_interest = 'place_cells' ## one of behav, aligned_minian, aligned_place_cells, lin_behav
data_type = 'S'
conversion = 2 / 0.06 ## 2cm per 0.06 radians

if not os.path.exists(fig_path):
    os.makedirs(fig_path)

xr.set_options(keep_attrs=True)

rs = RandomState(MT19937(SeedSequence(24601)))

### Example mouse.

In [ ]:
## Set mouse information
experiment = 'MultiCon_Imaging6'
mouse = 'mc58'
crossreg_str = 'first_days'
days_of_int = ['1', '6', '11', '16']
crossreg_path = f'../../../CircleTrack/{project_folder[0]}/{experiment}/output/cross_registration_results/circletrack_data/{mouse}'

cell_dict = {'mouse': [], 'group': [], 'sex': [], '0_percent': [], '1_percent': [], '2_percent': [], '3_percent': [], '4_percent': []}
## Select cells that are cross-registered across all four days
mappings_res = pd.read_pickle(pjoin(crossreg_path, f'mappings_meta_{centroid_distance}_{crossreg_str}.pkl'))['session'].dropna().reset_index(drop=True)

si_ar = np.zeros((mappings_res.shape[0], len(days_of_int)))
place_ar = np.zeros((mappings_res.shape[0], len(days_of_int)))
stab_ar = np.zeros((mappings_res.shape[0], len(days_of_int)))
for idx, d in enumerate(days_of_int):
    session = f'{mouse}_{data_type}_{d}.nc'
    exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/{mouse}/{data_type}')

    S = xr.open_dataset(pjoin(exp_path, session))[data_type] ## don't need to perform ctn.qc_matrix because aligned_place_cells already was qced and thresholded activity
    sdata = S.sel(unit_id=mappings_res[S.attrs['date']].to_numpy())
    si_ar[:, idx] = sdata['skaggs_info'].values
    place_ar[:, idx] = sdata['skaggs_place'].values
    stab_ar[:, idx] = sdata['odd_even'].values

place_cells = np.sum(place_ar, axis=1)
values, counts = np.unique(place_cells, return_counts=True)
norm_counts = (counts / np.sum(counts)) * 100

cell_dict['mouse'].append(mouse)
cell_dict['group'].append(S.attrs['group'])
cell_dict['sex'].append(S.attrs['sex'])
for idx in np.arange(0, 5):
    cell_dict[f'{idx}_percent'].append(norm_counts[idx])

### Combine across mice percentage of cells that are place cells across the four days.

In [ ]:
days_of_int = ['1', '6', '11', '16']

cell_dict = {'mouse': [], 'group': [], 'sex': [], '0_percent': [], '1_percent': [], '2_percent': [], '3_percent': [], '4_percent': []}
for experiment in os.listdir(dpath):
    if experiment not in experiment_folders:
        pass 
    else:
        exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
        for mouse in tqdm(os.listdir(exp_path)):
            if mouse in excluded_mice:
                pass 
            else:
                mpath = pjoin(exp_path, f'{mouse}/{data_type}')
                crossreg_path = f'../../../CircleTrack/{project_folder[0]}/{experiment}/output/cross_registration_results/circletrack_data/{mouse}'
                mappings_res = pd.read_pickle(pjoin(crossreg_path, f'mappings_meta_{centroid_distance}_{crossreg_str}.pkl'))['session'].dropna().reset_index(drop=True)

                si_ar = np.zeros((mappings_res.shape[0], len(days_of_int)))
                place_ar = np.zeros((mappings_res.shape[0], len(days_of_int)))
                for idx, d in enumerate(days_of_int):
                    session = f'{mouse}_{data_type}_{d}.nc'
                    S = xr.open_dataset(pjoin(mpath, session))[data_type] ## don't need to perform ctn.qc_matrix because aligned_place_cells already was qced and thresholded activity
                    sdata = S.sel(unit_id=mappings_res[S.attrs['date']].to_numpy())
                    si_ar[:, idx] = sdata['skaggs_info'].values
                    place_ar[:, idx] = sdata['skaggs_place'].values
                
                place_cells = np.sum(place_ar, axis=1)
                bins = np.arange(0, 6)
                H, _ = np.histogram(place_cells, bins=bins)
                norm_counts = (H / np.sum(H)) * 100

                cell_dict['mouse'].append(mouse)
                cell_dict['group'].append(S.attrs['group'])
                cell_dict['sex'].append(S.attrs['sex'])
                for idx in np.arange(0, 5):
                    cell_dict[f'{idx}_percent'].append(norm_counts[idx])
place_percent_df = pd.DataFrame(cell_dict)

In [ ]:
## Percentage of cells that are place cells in 0, 1, 2, 3, or 4 of the days
place_percent_df.groupby(['group'], as_index=False).agg({f'{x}_percent': ['mean', 'sem'] for x in np.arange(0, 5)})

In [ ]:
## Spatial information across the first days for cells that were place cells across all four days
si_ar[place_cells == 4]

### Plot high spatial information cells across time and location.

In [ ]:
## Sort spatial information from least to most SI
sorted_si = np.argsort(S['skaggs_info'].values)
uid = sorted_si[-1] ## select unit to visualize

print(f'Observed SI: {S['skaggs_info'].values[uid]}')
print(f'Shuffled SI: {S['shuffled_avg_si'].values[uid]}')
print('Place cell!') if S['skaggs_place'].values[uid] else print('Non-place cell!')
fig = pf.custom_graph_template(x_title='Time (s)', y_title='Activity (a.u.)')
fig.add_trace(go.Scattergl(x=S['behav_t'].values, y=S.values[uid, :], mode='lines', line_color='darkgrey'))
fig.show()

In [ ]:
## Plot heatmap of where the mouse was when the selected neuron fired
cscale = 'magma_r'
x_pos = S['x'].values
y_pos = S['y'].values
rw_one_x = np.mean(S['x'][(S['lick_port'] == S.attrs['reward_one'])])
rw_one_y = np.mean(S['y'][(S['lick_port'] == S.attrs['reward_one'])])
rw_two_x = np.mean(S['x'][(S['lick_port'] == S.attrs['reward_two'])])
rw_two_y = np.mean(S['y'][(S['lick_port'] == S.attrs['reward_two'])])

fig = pf.custom_graph_template(x_title='X Position', y_title='Y Position')
activity = S.values[uid]
H, x_edges, y_edges = np.histogram2d(x_pos, y_pos, bins=40, weights=activity)
norm_vals = H / np.max(H)
norm_vals[norm_vals == 0] = np.nan
fig.add_trace(go.Heatmap(x=x_edges, y=y_edges, z=norm_vals, colorscale=cscale, coloraxis='coloraxis1'))

fig.add_trace(go.Scattergl(x=[rw_one_x], y=[rw_one_y], mode='markers', marker_color='red', 
                        showlegend=False, name='Reward 1', marker_size=8))
fig.add_trace(go.Scattergl(x=[rw_two_x], y=[rw_two_y], mode='markers', marker_color='red', 
                        showlegend=False, name='Reward 2', marker_size=8))
fig.update_coloraxes(colorscale=cscale)
fig.show()

In [ ]:
## Plot scatterplot of firing rate vs spatial info
b = 1 ## in seconds
act_bin = ctn.bin_activity(S.values, bin_size_seconds=b, func=np.mean)
avg_act = np.mean(act_bin, axis=1) / b ## convert to Hz for any bin size

fig = pf.custom_graph_template(x_title='Firing Rate (Hz)', y_title='Spatial Information (bits/event)')
fig.add_trace(go.Scattergl(x=avg_act, y=S['skaggs_info'].values, mode='markers', marker_color='darkgrey',
                           marker_size=8, marker=dict(line=dict(width=1)), opacity=0.7))
fig.show()

### Cross-register cells between day 15 and day 16 to see what the highly stable cells were doing the day prior for an example mouse.

In [ ]:
## Set mouse information
experiment = 'MultiCon_Imaging6'
mouse = 'mc55'
crossreg_str = 'None'
days_of_int = ['15', '16']
crossreg_path = f'../../../CircleTrack/{project_folder[0]}/{experiment}/output/cross_registration_results/circletrack_data/{mouse}'
cell_dict = {'mouse': [], 'group': [], 'sex': [], 'day': [], 'low_stability_stability': [], 'high_stability_stability': [],
             'low_stability_si': [], 'high_stability_si': [], 'low_stability_fr': [], 'high_stability_fr': []}

## Load pairwise cross-registration for example mouse
mappings_res = pd.read_pickle(pjoin(crossreg_path, f'mappings_{centroid_distance}_{crossreg_str}.pkl'))['session']

## Get dates for subsetting
date_list = []
for d in days_of_int:
    session = f'{mouse}_{data_type}_{d}.nc'
    exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/{mouse}/{data_type}')
    S = xr.load_dataset(pjoin(exp_path, session))[data_type]
    date_list.append(S.attrs['date'])

## Select shared cells
shared_cells = mappings_res[date_list].dropna().reset_index(drop=True)
si_ar = np.zeros((shared_cells.shape[0], len(days_of_int)))
place_ar = np.zeros((shared_cells.shape[0], len(days_of_int)))
stab_ar = np.zeros((shared_cells.shape[0], len(days_of_int)))
fr_ar = np.zeros((shared_cells.shape[0], len(days_of_int)))
for idx, d in enumerate(days_of_int):
    session = f'{mouse}_{data_type}_{d}.nc'
    exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/{mouse}/{data_type}')
    S = xr.load_dataset(pjoin(exp_path, session))[data_type] ## don't need to perform ctn.qc_matrix because aligned_place_cells already was qced and thresholded activity
    sdata = S.sel(unit_id=shared_cells[date_list[idx]].to_numpy())
    si_ar[:, idx] = sdata['skaggs_info'].values
    place_ar[:, idx] = sdata['skaggs_place'].values
    stab_ar[:, idx] = sdata['odd_even'].values

    ## Calculate firing rates
    act_bin = ctn.bin_activity(sdata.values, bin_size_seconds=time_bin_size, func=np.mean)
    avg_act = np.mean(act_bin, axis=1) / time_bin_size ## convert to Hz for any bin size
    fr_ar[:, idx] = avg_act

## Calculate top 25th and bottom 25th percentile cut-offs
day16_highest_stab = np.percentile(stab_ar[:, 1][~np.isnan(stab_ar[:, 1])], q=75)
day16_lowest_stab = np.percentile(stab_ar[:, 1][~np.isnan(stab_ar[:, 1])], q=25)
high_indices = np.where(stab_ar[:, 1] > day16_highest_stab)[0]
low_indices = np.where(stab_ar[:, 1] <= day16_lowest_stab)[0]

for day_idx, day in enumerate(days_of_int):
    cell_dict['mouse'].append(mouse)
    cell_dict['group'].append(S.attrs['group'])
    cell_dict['sex'].append(S.attrs['sex'])
    cell_dict['day'].append(int(day))
    cell_dict['low_stability_stability'].append(np.nanmean(stab_ar[low_indices, day_idx]))
    cell_dict['high_stability_stability'].append(np.nanmean(stab_ar[high_indices, day_idx]))
    cell_dict['low_stability_si'].append(np.mean(si_ar[low_indices, day_idx]))
    cell_dict['high_stability_si'].append(np.mean(si_ar[high_indices, day_idx]))
    cell_dict['low_stability_fr'].append(np.mean(fr_ar[low_indices, day_idx]))
    cell_dict['high_stability_fr'].append(np.mean(fr_ar[high_indices, day_idx]))
cell_df = pd.DataFrame(cell_dict)

### Combine across mice day 15/16 stability percentiles.

In [ ]:
## Settings
crossreg_str = 'None'
days_of_int = ['15', '16']
cell_dict = {'mouse': [], 'group': [], 'sex': [], 'day': [], 'low_si_stability': [], 'high_si_stability': [],
             'low_si_si': [], 'high_si_si': [], 'low_si_fr': [], 'high_si_fr': []}

## Loop across mice and experiments
for experiment in os.listdir(dpath):
    if experiment not in experiment_folders:
        pass 
    else:
        exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
        for mouse in tqdm(os.listdir(exp_path)):
            if mouse in excluded_mice:
                pass 
            else:
                mpath = pjoin(exp_path, f'{mouse}/{data_type}')
                crossreg_path = f'../../../CircleTrack/{project_folder[0]}/{experiment}/output/cross_registration_results/circletrack_data/{mouse}'

                ## Load pairwise cross-registration for example mouse
                mappings_res = pd.read_pickle(pjoin(crossreg_path, f'mappings_{centroid_distance}_{crossreg_str}.pkl'))['session']

                ## Get dates for subsetting
                date_list = []
                for d in days_of_int:
                    session = f'{mouse}_{data_type}_{d}.nc'
                    exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/{mouse}/{data_type}')
                    S = xr.load_dataset(pjoin(exp_path, session))[data_type]
                    date_list.append(S.attrs['date'])

                ## Select shared cells
                shared_cells = mappings_res[date_list].dropna().reset_index(drop=True)
                si_ar = np.zeros((shared_cells.shape[0], len(days_of_int)))
                place_ar = np.zeros((shared_cells.shape[0], len(days_of_int)))
                stab_ar = np.zeros((shared_cells.shape[0], len(days_of_int)))
                fr_ar = np.zeros((shared_cells.shape[0], len(days_of_int)))
                for idx, d in enumerate(days_of_int):
                    session = f'{mouse}_{data_type}_{d}.nc'
                    exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/{mouse}/{data_type}')
                    S = xr.load_dataset(pjoin(exp_path, session))[data_type] ## don't need to perform ctn.qc_matrix because aligned_place_cells already was qced and thresholded activity
                    sdata = S.sel(unit_id=shared_cells[date_list[idx]].to_numpy())
                    si_ar[:, idx] = sdata['skaggs_info'].values
                    place_ar[:, idx] = sdata['skaggs_place'].values
                    stab_ar[:, idx] = sdata['odd_even'].values

                    ## Calculate firing rates
                    act_bin = ctn.bin_activity(sdata.values, bin_size_seconds=time_bin_size, func=np.mean)
                    avg_act = np.mean(act_bin, axis=1) / time_bin_size ## convert to Hz for any bin size
                    fr_ar[:, idx] = avg_act

                ## Calculate top 25th and bottom 25th percentile cut-offs
                day16_highest_si = np.percentile(si_ar[:, 1][~np.isnan(si_ar[:, 1])], q=75)
                day16_lowest_si = np.percentile(si_ar[:, 1][~np.isnan(si_ar[:, 1])], q=25)
                high_indices = np.where(si_ar[:, 1] > day16_highest_si)[0]
                low_indices = np.where(si_ar[:, 1] <= day16_lowest_si)[0]

                for day_idx, day in enumerate(days_of_int):
                    cell_dict['mouse'].append(mouse)
                    cell_dict['group'].append(S.attrs['group'])
                    cell_dict['sex'].append(S.attrs['sex'])
                    cell_dict['day'].append(int(day))
                    cell_dict['low_si_stability'].append(np.nanmean(stab_ar[low_indices, day_idx]))
                    cell_dict['high_si_stability'].append(np.nanmean(stab_ar[high_indices, day_idx]))
                    cell_dict['low_si_si'].append(np.mean(si_ar[low_indices, day_idx]))
                    cell_dict['high_si_si'].append(np.mean(si_ar[high_indices, day_idx]))
                    cell_dict['low_si_fr'].append(np.mean(fr_ar[low_indices, day_idx]))
                    cell_dict['high_si_fr'].append(np.mean(fr_ar[high_indices, day_idx]))
cell_df = pd.DataFrame(cell_dict)

In [ ]:
## Plot day 15 to day 16's stability, spatial info, or firing rate separate for low-stable or high-stable cells on day 16
yvar = 'fr'
fig = pf.custom_graph_template(x_title='Days', y_title='', titles=['Low SI Cells', 'High SI Cells'], rows=1, columns=2,
                               shared_y=True, shared_x=True, width=800)

for group in ['Two-context', 'Multi-context']:
    gdata = cell_df[cell_df['group'] == group]
    for mouse in gdata['mouse'].unique():
        mdata = gdata[gdata['mouse'] == mouse]

        for col in [1, 2]:
            if yvar == 'stability':
                y = ['low_si_stability', 'high_si_stability']
                y_title = 'Stability'
                y_range = [-0.05, 1]
            elif yvar == 'si':
                y = ['low_si_si', 'high_si_si']
                y_title = 'Spatial Information (bits/event)'
                y_range = [0, 9]
            elif yvar == 'fr':
                y = ['low_si_fr', 'high_si_fr']
                y_title = 'Firing Rate (Hz)'
                y_range = [0, 0.015]

            fig.add_trace(go.Scattergl(x=mdata['day'].astype(str), y=mdata[y[col - 1]], mode='lines+markers', marker_color=ce_colors_dict[group],
                                       marker=dict(line=dict(width=1.5, color='black')), name=group, legendgroup=group,
                                       marker_size=9, showlegend=False, opacity=0.7), row=1, col=col)
fig.update_yaxes(title=y_title, col=1)
fig.update_yaxes(range=y_range)
fig['data'][0]['showlegend'] = True
fig['data'][20]['showlegend'] = True
fig.show()
fig.write_image(pjoin(fig_path, f'low_high_si_days15_16_{yvar}.png'), width=800, height=500)

### Look at reward over-representation between highly spatially informative cells shared across two days.

In [ ]:
## Settings
only_running = True
correct_dir = True
cell_type = 'all_cells'
keep_cells = 'spatial_info'
window_size = 10 ## number of spatial bins
rel_bins = np.arange(-(bin_size * window_size), (bin_size * window_size) + bin_size, bin_size)

In [ ]:
## Settings
crossreg_str = 'None'
days_of_int = ['15', '16']
reference_day = days_of_int[0]
output_dict = {'mouse': [], 'group': [], 'sex': [], 'day': [], 'session': [], 'relative_bin': [], 'proportion_one': [], 'proportion_two': []}
reward_bins = np.arange(0, 6.28 + reward_bin_size, reward_bin_size)

## Loop across mice and experiments
for experiment in os.listdir(dpath):
    if experiment not in experiment_folders:
        pass 
    else:
        exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
        for mouse in tqdm(os.listdir(exp_path)):
            if mouse in excluded_mice:
                pass 
            else:
                mpath = pjoin(exp_path, f'{mouse}/{data_type}')
                crossreg_path = f'../../../CircleTrack/{project_folder[0]}/{experiment}/output/cross_registration_results/circletrack_data/{mouse}'

                ## Load pairwise cross-registration for example mouse
                mappings = pd.read_pickle(pjoin(crossreg_path, f'mappings_{centroid_distance}_{crossreg_str}.pkl'))['session']

                ## Get dates for subsetting cross-registration results
                try:
                    date_list = []
                    for d in days_of_int:
                        session = f'{mouse}_{data_type}_{d}.nc'
                        exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/{mouse}/{data_type}')
                        S = xr.load_dataset(pjoin(exp_path, session))[data_type]
                        date_list.append(S.attrs['date'])

                    ## Select shared cells
                    shared_cells = mappings[date_list].dropna().reset_index(drop=True)

                    ## Get the reference day's unit_ids
                    ref_idx = np.where(np.array(days_of_int) == reference_day)[0][0]
                    S = xr.load_dataset(pjoin(exp_path, f'{mouse}_{data_type}_{reference_day}.nc'))[data_type]
                    sdata = S.sel(unit_id=shared_cells[date_list[ref_idx]].to_numpy()) ## use the reference day to index date list
                    sdata = sdata[sdata['minimum_trial_activity_met'], :] ## only use cells that meet activity threshold
                    if keep_cells == 'spatial_info':
                        si_values = sdata['skaggs_info'].values
                        si_percentile = np.percentile(si_values, q=75)
                        sub_uids = sdata['unit_id'][si_values > si_percentile].values
                    elif keep_cells == 'stability':
                        stab_values = sdata['odd_even'].values[~np.isnan(sdata['odd_even'].values)] ## remove any NaNs
                        stab_percentile = np.percentile(stab_values, q=75)
                        sub_uids = sdata['unit_id'][sdata['odd_even'].values > stab_percentile].values ## select cells above the 75th percentile
                    
                    ## Loop through all days and use the cells selected above
                    for idx, d in enumerate(days_of_int):
                        session = f'{mouse}_{data_type}_{d}.nc'
                        exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/{mouse}/{data_type}')
                        S = xr.load_dataset(pjoin(exp_path, session))[data_type] ## don't need to perform ctn.qc_matrix because aligned_place_cells already was qced and thresholded activity
                        sdata = S.sel(unit_id=shared_cells[S.attrs['date']][shared_cells[date_list[ref_idx]].isin(sub_uids)].to_numpy())

                        neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                                        velocity_thresh=velocity_thresh)
                        ## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
                        population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
                        active_cells = np.sum(population_activity, axis=0) != 0
                        population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
                        tuning_curves = pc.get_tuning_curves(population_activity, occupancy)
                        tuning_curves = tuning_curves.T ## cells x spatial bin

                        ## Get reward positions
                        reward_one_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_one']) & (sdata['correct_dir'])].values)
                        reward_two_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_two']) & (sdata['correct_dir'])].values)

                        ## Make Reward 1 the first rewarding port the mouse got water from
                        first_rew = sdata['lick_port'][sdata['water']].values[0]
                        if first_rew == sdata.attrs['reward_one']:
                            first_rw_pos = reward_one_pos 
                            second_rw_pos = reward_two_pos
                        else:
                            first_rw_pos = reward_two_pos 
                            second_rw_pos = reward_one_pos

                        h_shift_one, h_shift_two, H, xbin, mid_bin, rel_rw_one, rel_rw_two = pc.pf_relative_reward(tuning_curves, first_rw_pos, second_rw_pos, bins=bins[:-1], 
                                                                                                                   reward_bins=reward_bins, proportion=True)
                        indices = np.arange(mid_bin - window_size, mid_bin + window_size + 1)

                        for i, rbin in enumerate(rel_bins):
                            output_dict['mouse'].append(mouse)
                            output_dict['group'].append(sdata.attrs['group'])
                            output_dict['sex'].append(sdata.attrs['sex'])
                            output_dict['day'].append(int(re.search('[0-9]+', session.split('_')[2])[0]))
                            output_dict['session'].append(sdata.attrs['session_two'])
                            output_dict['relative_bin'].append(rbin)
                            output_dict['proportion_one'].append(h_shift_one[indices[i]])
                            output_dict['proportion_two'].append(h_shift_two[indices[i]])
                except:
                    pass

In [ ]:
## Plot the proportion of place fields for specific days
sub_rel_bins = 8
shared_rel_df = pd.DataFrame(output_dict)
shared_rel_df['relative_bin'] = shared_rel_df['relative_bin'] * conversion ## convert from radians to cm
shared_rel_df = shared_rel_df[(shared_rel_df['relative_bin'] >= -sub_rel_bins - 0.5) & (shared_rel_df['relative_bin'] <= sub_rel_bins)]
avg_df = shared_rel_df.groupby(['group', 'day', 'relative_bin'], as_index=False).agg({'proportion_one': ['mean', 'sem'], 'proportion_two': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='Reward Distance (cm)', y_title='', rows=1, columns=2, width=1000, titles=[f'Day {days_of_int[0]}', f'Day {days_of_int[1]}'],
                               shared_x=True, shared_y=True, master_axes=False)

for group in ['Two-context', 'Multi-context']:
    for idx, day in enumerate(days_of_int):
        sub = avg_df[(avg_df['group'] == group) & (avg_df['day'] == int(day))]
        combined_data = {'relative_bin': [], 'prob': [], 'sem': []}
        for rbin in sub['relative_bin'].unique():
            r_sub = sub[sub['relative_bin'] == rbin]
            combined_data['relative_bin'].append(rbin)
            combined_data['prob'].append(np.sum((r_sub['proportion_one']['mean'].values[0], r_sub['proportion_two']['mean'].values[0])))
            combined_data['sem'].append(np.sum((r_sub['proportion_one']['sem'].values[0], r_sub['proportion_two']['sem'].values[0])))
        fig.add_trace(go.Scattergl(x=combined_data['relative_bin'], y=combined_data['prob'], showlegend=False, mode='lines+markers',
                                   legendgroup=group, name=group, marker_color=ce_colors_dict[group], marker_line_width=1.5, marker_size=7,
                                   marker_line_color='black', error_y=dict(type='data', array=combined_data['sem'])), row=1, col=idx + 1)
fig.add_vrect(x0=-2, x1=2, fillcolor=chance_color, opacity=0.5, layer="below", line_width=0)
fig.update_yaxes(title='Proportion Place Fields', col=1)
fig.update_yaxes(range=[-0.01, 0.25])
fig.update_xaxes(range=[-sub_rel_bins - 2, sub_rel_bins + 2])
fig['data'][0]['showlegend'] = True
fig['data'][2]['showlegend'] = True                          
fig.show()
# fig.write_image(pjoin(fig_path, f'day{days_of_int[0]}_day{days_of_int[1]}_ref_day_{reference_day}_shared_cells_top25_{keep_cells}_rw_representation_combined.png'), width=1000, height=500)